# 💬 生成式AI應用開發｜第05週｜150分鐘修訂版
## 手動對話紀錄、Streaming 與可重設聊天工具
承接第四周訊息列表 few-shot，將人工範例替換成實際 API 回答；銜接第七周最近五輪 history。
本檔另存修訂，原教材保留。先備：函式、list/dict、for、切片、role/content、函式作為參數。

> **學生版**：練習A的訊息組裝／reset、練習B串流迴圈保留TODO，類別封裝與重要事實為選讀。

## 🎯 學習目標
1. 分清人工 few-shot 範例與實際對話 history。
2. 以手動 user/assistant 配對支援追問，每輪重送 instructions。
3. 以 reset／最近N輪控制下一輪的輸入，觀察 usage。
4. 比較 response ID 與手動 history 的狀態保存方式。
5. 接收 delta、累積完整文字、確認 completed，失敗不寫入正式紀錄。
6. 用固定測試與實際請求內容驗證聊天行為。

## 🗺️ 150分鐘有效授課流程（休息另計）
| 分鐘 | 活動 | 學生成果 |
|---|---|---|
| 0–15 | 第四周銜接與獨立請求 | 區分範例與歷史 |
| 15–40 | 手動history、兩輪追問 | 訊息組裝、完成回答配對 |
| 40–55 | response ID比較 | 說明保存方式與取捨 |
| 55–80 | 必做A：reset、最近N輪 | 離線請求與有限視窗 |
| 80–105 | Streaming事件、模擬片段 | 看懂delta與completed |
| 105–135 | 必做B：串流與失敗保護 | 成功保存與失敗測試 |
| 135–150 | 固定測試、同儕檢查 | 紀錄與修正理由 |
類別、長期重要事實與互動CLI為延伸；不是本週必做門檻。
若API排障超過5分鐘，先用虛構回應完成離線契約檢查。

## 1️⃣ 環境與成本
金鑰放Colab Secrets的OPENAI_API_KEY；本機用環境變數，勿提交金鑰。
無金鑰可完成函式定義、訊息檢查及假事件測試。所有付費旗標預設False。
安裝cell會連線安裝SDK，但不呼叫模型；後面的假事件測試不連線。

In [ ]:
# Colab runtime重新開啟時安裝SDK；本機也可使用已安裝的課堂環境。
!pip install -q --upgrade openai

In [ ]:
import os
from openai import OpenAI

# Colab 優先讀取 Secrets；本機執行時則改用既有環境變數。

try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    # 非 Colab 環境會走到這裡，不需要中斷 notebook。
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")


# 沒有金鑰仍能做離線組裝與假事件練習；付費入口才檢查 client。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print("本週模型：", MODEL)
print("已讀取金鑰（不顯示）" if client else "離線模式：尚未設定金鑰")

# 所有真實 API 示範與互動均預設關閉，分段確認後才開啟。
RUN_STATELESS_DEMO = False
RUN_MANUAL_DEMO = False
RUN_ID_COMPARISON = False
RUN_PRACTICE_A = False
RUN_STREAM_DEMO = False
RUN_PRACTICE_B = False
RUN_LIVE_TESTS = False

In [ ]:
import json

CHAT_INSTRUCTIONS = """
# Identity
你是生成式 AI 應用開發課程的助教。
# Instructions
- 使用繁體中文，先直接回答，再補充必要說明。
- 回答控制在 150 字內；不知道時清楚說明限制。
- 不要聲稱記得目前對話以外的資料。
"""



def require_text(text, name="user_input"):
    """空白輸入在本機擋下，不產生付費請求。"""
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f"{name} 必須是非空白字串。")


def require_client(api_client):
    if api_client is None:
        raise RuntimeError("請設定 OPENAI_API_KEY，重新執行環境 cell。")


def validated_output(response):
    """HTTP 請求成功仍可能沒有可用答案；完成、拒答、空文字要分開檢查。"""
    if response.status != "completed":
        raise RuntimeError(f"回應未完成：{response.status}；不保存部分內容。")
    for item in response.output or []:
        for part in getattr(item, "content", []) or []:
            if getattr(part, "type", None) == "refusal":
                raise RuntimeError("模型拒答；本輪不寫入對話紀錄。")
    require_text(response.output_text, "output_text")
    return response.output_text


def create_response(user_input, previous_response_id=None, store=True, api_client=None):
    """供獨立請求與 response ID 比較示範；主線使用下方手動 history。"""
    require_text(user_input)
    active_client = client if api_client is None else api_client
    require_client(active_client)
    request = dict(model=MODEL, instructions=CHAT_INSTRUCTIONS, input=user_input, store=store)
    if previous_response_id:
        request["previous_response_id"] = previous_response_id
    try:
        response = active_client.responses.create(**request)
    except Exception as exc:
        raise RuntimeError(f"API 請求失敗：{exc}") from exc
    validated_output(response)
    return response

### 從第四周到第五周
| 項目 | 第四周 | 第五周 |
|---|---|---|
| assistant內容 | 人工指定範例答案 | API實際完成的回答 |
| 用途 | 示範任務模式 | 提供前文以支援追問 |
| 保存 | 固定範例 | 隨互動增加的本機history |
規則仍放instructions；看到Notebook有舊輸出，不代表模型收到它。
先預測第二輪要看到哪些訊息，再印出送出的input。

## 2️⃣ 獨立請求：為何忘記前文？
兩次請求若沒有附history或response ID，第二次看不到第一輪。
觀察是否拒絕猜測姓名；即使碰巧猜對，也不能證明具有記憶。

In [ ]:
if RUN_STATELESS_DEMO:
    first = create_response("我叫小安，專題是校園活動推薦。", store=False)
    second = create_response("我的姓名與專題是什麼？", store=False)
    print(first.output_text)
    print(second.output_text)

## 3️⃣ 手動history：先做最小的兩輪流程
第一輪只送user；收到有效完成回答後，保存user與assistant。
第二輪把已完成配對加上新user送入input。固定instructions每輪提供。
下列最小示範先不裁切，避免一次加入太多概念；練習A再實作有限視窗。
本週為單一文字回覆的簡化history；需要保留推理／工具項目的正式流程應保存必要的完整output項目。

In [ ]:
if RUN_MANUAL_DEMO:
    require_client(client)
    demo_history = []
    for question in ["我叫小安，專題是校園活動推薦。", "請重述我的姓名與專題。"]:
        request_input = demo_history + [{"role": "user", "content": question}]
        print("本輪實際input：", request_input)
        response = client.responses.create(
            model=MODEL, instructions=CHAT_INSTRUCTIONS, input=request_input, store=False,
        )
        answer = validated_output(response)
        demo_history.extend([
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
        print(answer)

### ✍️ 預測與觀察
| 回合 | 預測input訊息數 | 實際input | 是否依前文作答 | 有無編造 |
|---|---|---|---|---|
| 第一輪 |  |  |  |  |
| 第二輪 |  |  |  |  |
判讀：第一輪1則，第二輪3則；第二輪assistant是第一輪真正收到的答案。
回答措辭可以不同，姓名／主題須符合前文。

## 4️⃣ response ID作比較（不另寫一套完整聊天工具）
| 方法 | 本機保存 | 每輪傳送 | 本週定位 |
|---|---|---|---|
| 手動history | 完成的訊息配對 | 最近N輪＋本次問題 | 核心實作、第七周銜接 |
| previous_response_id | 前輪response.id | 前輪ID＋本次問題 | 比較示範 |
| Conversations API | conversation ID | 依API管理持久對話 | 概念選讀 |
response ID串接仍每輪提供instructions；鏈中舊輸入仍可能計入input tokens。
本比較採store=True；本機reset不等於刪除平台已保存response。
store=False也不代表資料沒有傳送或處理；只使用虛構資料。

In [ ]:
if RUN_ID_COMPARISON:
    id_first = create_response("我叫小安，專題是校園活動推薦。", store=True)
    id_second = create_response("請重述姓名與專題。", previous_response_id=id_first.id, store=True)
    print(id_first.id, "→", id_second.id)
    print(id_second.output_text)

## 5️⃣ 必做A：本機state、最近N輪與reset
state是dict，不需先學類別或繼承。history只保存已完成配對；本次user另加到請求。
`max_turns=5`代表最近10則已完成訊息，再加本次user共最多11則。
切片控制送進模型的context；本機history仍可保留全部，兩者不能混淆。
reset清除本機history及本session統計，不會抹除平台紀錄或帳單。

In [ ]:
def new_chat_state():
    """本機保存已完成 history、成功回合使用量與成功請求數。"""
    return {
        "history": [],
        "usage": {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
        "completed_requests": 0,
    }


def history_size(messages):
    """字元數方便觀察，不能當成精準 token 數或費用。"""
    return {"訊息數": len(messages), "總字元數": sum(len(m["content"]) for m in messages)}


def save_completed_turn(state, user_text, answer, response):
    """所有回應檢查通過後才一次寫入 user/assistant 配對。"""
    # 先準備 usage，避免格式讀取錯誤後留下只更新一半的 state。
    usage = getattr(response, "usage", None)
    new_usage = {
        key: state["usage"][key] + (getattr(usage, key, 0) or 0)
        for key in state["usage"]
    }
    state["history"].extend([
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": answer},
    ])
    state["usage"] = new_usage
    state["completed_requests"] += 1

In [ ]:
def build_chat_input(user_text, history, max_turns=5):
    """A1：只選最近完整N輪，再附上本次user，不修改原history。"""
    require_text(user_text)
    if type(max_turns) is not int or max_turns <= 0:
        raise ValueError("max_turns必須是正整數。")
    # TODO A1：取history最近2*max_turns則，複製訊息後附上本次user，回傳列表。
    # 提示：history中每輪都是user/assistant配對；不能直接append到原列表。
    raise NotImplementedError("請完成最近N輪組裝。")


def reset_chat(state):
    """A2：清空本機state，原dict物件仍可被介面使用。"""
    # TODO A2：clear state，再用new_chat_state()的內容更新。
    raise NotImplementedError("請完成reset。")

### 先離線檢查，再呼叫API
若自行加強輸入驗證，可檢查history長度為偶數、角色交錯、content非空白。
不要先把新user寫進state；如果API失敗，應保持原state。
下方提供共用非串流入口，學生專注A1/A2與觀察請求。

In [ ]:
def ask_chat_once(state, user_text, max_turns=5, instructions=CHAT_INSTRUCTIONS, api_client=None):
    """手動 history 的一次問答；完成後保存，失敗不改 state。"""
    require_text(user_text)
    require_text(instructions, "instructions")
    messages = build_chat_input(user_text, state["history"], max_turns)
    active_client = client if api_client is None else api_client
    require_client(active_client)
    try:
        response = active_client.responses.create(
            model=MODEL, instructions=instructions, input=messages, store=False,
        )
    except Exception as exc:
        raise RuntimeError(f"聊天失敗，狀態未更新：{exc}") from exc
    answer = validated_output(response)
    save_completed_turn(state, user_text, answer, response)
    return answer

In [ ]:
RUN_INPUT_CHECK = False
if RUN_INPUT_CHECK:
    sample_history = [
        {"role": "user", "content": "我叫小安"}, {"role": "assistant", "content": "你好小安"},
        {"role": "user", "content": "我喜歡Python"}, {"role": "assistant", "content": "了解"},
    ]
    trimmed = build_chat_input("我叫什麼？", sample_history, max_turns=1)
    assert trimmed == sample_history[-2:] + [{"role": "user", "content": "我叫什麼？"}]
    assert len(sample_history) == 4
    print("真正送入的input：", trimmed)
    print("規模：", history_size(trimmed))
    scratch = new_chat_state()
    scratch["history"] = sample_history.copy()
    reset_chat(scratch)
    assert scratch == new_chat_state()

In [ ]:
if RUN_PRACTICE_A:
    practice_state = new_chat_state()
    for question in ["我叫小安，專題是課程FAQ。", "我的專題是什麼？"]:
        print("本輪input：", build_chat_input(question, practice_state["history"], 5))
        print(ask_chat_once(practice_state, question))
    print("累積usage：", practice_state["usage"])
    reset_chat(practice_state)
    print("reset後input：", build_chat_input("我的專題是什麼？", practice_state["history"]))
    print(ask_chat_once(practice_state, "我的專題是什麼？"))

### ✍️ 狀態觀察表
| 操作 | 本機history則數 | 實際input則數 | 成功請求數 | usage | 判讀 |
|---|---|---|---|---|---|
| 第一輪 |  |  |  |  |  |
| 第二輪 |  |  |  |  |  |
| reset後 |  |  |  |  |  |
| 裁切最近1輪 |  |  |  |  |  |
「送出的資料不含姓名」是reset／裁切的證據；模型猜對或猜錯本身不是可靠證據。
usage只累計成功回合，不代表所有帳單費用；失敗／中斷可能仍有費用，須核對平台用量。

## 6️⃣ Streaming：先模擬，再寫事件迴圈
| 事件 | 做什麼 | 是否寫入正式history |
|---|---|---|
| response.created | 知道串流開始 | 否 |
| response.output_text.delta | 顯示片段、累積文字、增加計數 | 否 |
| response.completed | 驗證完成文字、取得usage | 通過檢查後才保存 |
| error／response.failed／response.incomplete | 提示失敗、停止 | 否 |
| 結束卻沒有completed | 視為未完成 | 否 |
delta數不是token數，也不是字數。收到片段不表示整輪成功。
畫面可能已有部分文字；失敗時須標示中斷，不能把它當完整答案保存。

In [ ]:
# 不付費的小示範：迴圈同時顯示與累積，不需要callback或nonlocal。
simulated_chunks = ["你好", "，", "我可以幫忙。"]
collected = []
delta_count = 0
for delta in simulated_chunks:
    print(delta, end="", flush=True)
    collected.append(delta)
    delta_count += 1
print("\n完整文字：", "".join(collected), "｜片段數：", delta_count)

### 接回第四周：傳入函式，由事件迴圈呼叫
第四周把分類函式交給測試器；此處把顯示函式交給串流器。
`on_delta=print_delta`傳入函式；收到delta才執行`callback(event.delta)`。
`list.append`也能當callback，用來收集已顯示片段。計數仍留在事件迴圈中，無須nonlocal。

## 7️⃣ 必做B：串流事件與成功保存
下方保留驗證／參數骨架，完成事件分支與回傳；重用A的state與訊息組裝。
假事件先測成功、失敗及沒有completed，再開啟真實API。

In [ ]:
def print_delta(delta):
    """具名 callback：只顯示片段，不負責保存正式對話紀錄。"""
    print(delta, end="", flush=True)


def stream_response(messages, instructions=CHAT_INSTRUCTIONS, on_delta=None, api_client=None):
    """手動 history 串流入口；回傳完整文字、完成response、delta數。"""
    require_text(instructions, "instructions")
    if not isinstance(messages, list) or not messages:
        raise ValueError("messages 必須是非空列表。")
    active_client = client if api_client is None else api_client
    require_client(active_client)
    chunks = []
    chunk_count = 0
    completed_response = None
    # 函式參數的呼叫方式與第四周的測試器相同；不需要 nonlocal。
    callback = print_delta if on_delta is None else on_delta

    # TODO B1：啟動 stream=True、store=False，逐筆讀取 event。
    # TODO B2：delta 時累積文字、增加 chunk_count、呼叫 callback。
    # TODO B3：completed 保存 event.response；error/failed/incomplete 拋出 RuntimeError。
    # TODO B4：沒有 completed 時拒絕；用 validated_output 檢查完成回答。
    # TODO B5：join chunks，確認與完成文字相同，再回傳三個值。
    # 提示：非文字事件可忽略；任何失敗都不能回傳部分文字當作成功。
    raise NotImplementedError("請完成串流事件迴圈。")

In [ ]:
def ask_chat_stream(state, user_text, max_turns=5, instructions=CHAT_INSTRUCTIONS,
                    on_delta=None, api_client=None):
    """重用非串流的訊息組裝及成功保存邏輯，只有取得回答的方式不同。"""
    messages = build_chat_input(user_text, state["history"], max_turns)
    full_text, response, chunk_count = stream_response(
        messages, instructions, on_delta=on_delta, api_client=api_client,
    )
    save_completed_turn(state, user_text, full_text, response)
    return full_text, chunk_count

In [ ]:
if RUN_STREAM_DEMO:
    demo_state = new_chat_state()
    demo_text, demo_count = ask_chat_stream(demo_state, "用兩句話說明API。")
    print("\n完整文字：", demo_text, "delta數：", demo_count)
    print("正式history：", demo_state["history"])

if RUN_PRACTICE_B:
    stream_state = new_chat_state()
    for question in ["我叫小安，專題是課程FAQ。", "請重述我的姓名與專題。"]:
        print("本輪input：", build_chat_input(question, stream_state["history"]))
        text, count = ask_chat_stream(stream_state, question)
        print("\n片段數：", count, "｜完整字元數：", len(text))
    print("usage：", stream_state["usage"])

## 8️⃣ 固定測試：先看契約，再看模型品質
五個核心條件：前文帶入、reset、最近N輪、空白不呼叫、串流完成才保存。
假回應驗證程式流程；不能證明模型會正確回答。真實API另測姓名與主題、資訊不足及格式。
成功答案不需逐字相同；狀態與失敗保護則要有明確可檢查的條件。

In [ ]:
from types import SimpleNamespace


def fake_response(text="你好！", status="completed", refusal=False):
    """虛構回應只檢查程式流程，無法驗證模型品質或真實SDK行為。"""
    content = [SimpleNamespace(type="refusal")] if refusal else []
    return SimpleNamespace(
        id="offline-response", status=status, output_text=text,
        output=[SimpleNamespace(content=content)],
        usage=SimpleNamespace(input_tokens=10, output_tokens=3, total_tokens=13),
    )


def make_fake_client(payload, requests):
    # callback 以具名函式實作，不引入 class、lambda 或 nonlocal。
    def create(**kwargs):
        requests.append(kwargs)
        return payload
    return SimpleNamespace(responses=SimpleNamespace(create=create))


def make_fake_events(kind="completed"):
    events = [
        SimpleNamespace(type="response.created"),
        SimpleNamespace(type="response.output_text.delta", delta="你好"),
        SimpleNamespace(type="response.output_text.delta", delta="！"),
    ]
    if kind == "completed":
        events.append(SimpleNamespace(type="response.completed", response=fake_response()))
    elif kind != "missing":
        events.append(SimpleNamespace(type=kind))
    return events

In [ ]:
def run_offline_checks():
    """完成練習後先驗證五個契約；不使用 API key，也不付費。"""
    rows = []
    requests = []
    state = new_chat_state()
    fake = make_fake_client(fake_response("姓名是小安"), requests)
    ask_chat_once(state, "我叫小安", api_client=fake)
    ask_chat_once(state, "我叫什麼？", api_client=fake)
    assert requests[1]["input"] == state["history"][:3]
    assert requests[1]["instructions"] == CHAT_INSTRUCTIONS
    rows.append("PASS：第二輪包含真實保存的前文")

    reset_chat(state)
    new_input = build_chat_input("我叫什麼？", state["history"])
    assert new_input == [{"role": "user", "content": "我叫什麼？"}]
    assert state["completed_requests"] == 0 and state["usage"]["total_tokens"] == 0
    rows.append("PASS：reset 後請求沒有舊姓名")

    history = [
        {"role": "user", "content": "舊問題"}, {"role": "assistant", "content": "舊答案"},
        {"role": "user", "content": "新問題"}, {"role": "assistant", "content": "新答案"},
    ]
    assert build_chat_input("追問", history, 1) == history[-2:] + [{"role": "user", "content": "追問"}]
    rows.append("PASS：最近N輪保持完整配對")

    before = len(requests)
    try:
        ask_chat_once(state, " ", api_client=fake)
    except ValueError:
        pass
    else:
        raise AssertionError("空白輸入沒有被擋下。")
    assert len(requests) == before and state["history"] == []
    rows.append("PASS：空白輸入不呼叫API")

    displayed = []
    stream_fake = make_fake_client(make_fake_events(), requests)
    text, count = ask_chat_stream(state, "你好", on_delta=displayed.append, api_client=stream_fake)
    assert text == "你好！" and count == 2 and "".join(displayed) == text
    assert len(state["history"]) == 2
    snapshot = json.loads(json.dumps(state))
    for kind in ("error", "response.failed", "response.incomplete", "missing"):
        bad = make_fake_client(make_fake_events(kind), requests)
        try:
            ask_chat_stream(state, "失敗回合", on_delta=displayed.append, api_client=bad)
        except RuntimeError:
            pass
        else:
            raise AssertionError(f"未拒絕：{kind}")
        assert state == snapshot
    rows.append("PASS：串流成功才保存；失敗／缺完成事件維持原狀態")
    return rows


RUN_OFFLINE_CHECKS = False
if RUN_OFFLINE_CHECKS:
    for row in run_offline_checks():
        print(row)

In [ ]:
def run_live_chat_checks():
    """三次真實請求，印出請求與回答供人工覆核，不用猜測式答案assert。"""
    state = new_chat_state()
    for question in ["我叫小安，專題是課程FAQ。", "我的姓名與專題是什麼？"]:
        print("input：", build_chat_input(question, state["history"]))
        print(ask_chat_once(state, question))
    reset_chat(state)
    print("reset後input：", build_chat_input("我的姓名是什麼？", state["history"]))
    print(ask_chat_once(state, "我的姓名是什麼？"))
    print("reset後usage：", state["usage"])

if RUN_LIVE_TESTS:
    run_live_chat_checks()

### 🧪 測試與修訂紀錄
| 條件 | 預期請求／狀態 | 實際 | 通過 | 修改理由 |
|---|---|---|---|---|
| 第二輪追問 | 帶第一輪user/assistant |  |  |  |
| reset | 不帶舊資料 |  |  |  |
| 最近N輪 | 完整配對＋本次user |  |  |  |
| 空白 | API請求數不變 |  |  |  |
| 串流中斷 | history、usage、成功數不變 |  |  |  |
| 真實回答 | 姓名／主題符合前文，reset後不聲稱記得 |  |  |  |
同儕覆核：對方能否從input指出模型看得到與看不到的內容？失敗是否留下半套紀錄？

## 9️⃣ 課後整合：簡易CLI骨架（選做）
課堂先用固定問題列表，避免互動input卡住「全部執行」。
完成A/B後才開啟CLI，支援/reset與/exit；失敗時保留原對話，提示重試。

In [ ]:
def run_console_chat():
    """重用手動history串流函式，不另寫第二套聊天邏輯。"""
    state = new_chat_state()
    while True:
        user_text = input("\n你：").strip()
        if user_text == "/exit":
            break
        if user_text == "/reset":
            reset_chat(state)
            print("本機對話已清空。")
            continue
        try:
            ask_chat_stream(state, user_text)
            print()
        except (ValueError, RuntimeError) as exc:
            print(f"\n本輪未成功：{exc}，可重試。")

# 完成練習並確認成本後，手動取消下一行註解；每輪都會付費。
# run_console_chat()

## 🔎 進階選讀：重要事實與類別封裝
重要偏好可另存在本機state；如需提供模型，可放在清楚標示的資料區。
使用者提供的事實不是開發者規則，不應直接提升成可覆蓋instructions的指令。
若本機未寫檔或存資料庫，重啟程式後就不會保留；不稱為永久記憶。
類別只是把同一套函式與state包在一起，不是本週必做。下例不使用繼承、super或nonlocal。

選做任務：另建preferred_language與project_topic兩項本機資料，在使用者資料區附上它們；
裁切一般history後印出input，確認仍含選定資料；再說明reset是否清除它們。
不得將使用者提供的任意文字當作高層規則；不在此要求額外付費實驗。


In [ ]:
class ManualChatSession:
    """選讀：state成為物件屬性，核心函式維持原樣。"""
    def __init__(self, instructions=CHAT_INSTRUCTIONS, max_turns=5):
        self.instructions = instructions
        self.max_turns = max_turns
        self.state = new_chat_state()

    def ask(self, user_text):
        return ask_chat_once(self.state, user_text, self.max_turns, self.instructions)

    def ask_stream(self, user_text):
        return ask_chat_stream(self.state, user_text, self.max_turns, self.instructions)

    def reset(self):
        reset_chat(self.state)

# 只定義類別，不在此呼叫API。讀懂函式版後再比較self.state與普通state。

## ✅ 完成檢核與後續銜接
- [ ] 能說明人工範例、實際history與response ID的差別。
- [ ] 完成A：最近N輪與reset，印出真正送出的input。
- [ ] 完成B：delta累積、完成驗證、失敗不更新history。
- [ ] 固定instructions每輪送出，usage不等於完整帳單。
- [ ] 通過離線五項契約，記錄真實API尚未驗證的部分。
- [ ] 能區分模型答錯、輸入／輸出契約錯誤及API失敗。
第六周仍教Structured Outputs與資料抽取；第五周不提前塞入JSON Schema。
第七周把同一history放入st.session_state，最近五輪加本次user送入模型。

## 📝 課後任務
選課程FAQ、學習助教或校園資訊，繳交instructions、三輪含追問的對話、reset前後input、
最近N輪截取結果、串流片段與完整文字、五項離線測試、真實測試紀錄與usage。
提供150字反思：哪種狀態保存在本機？何時才能保存回答？失敗時如何重試？
CLI、類別與重要事實為延伸，可保留選讀，不要求全部完成。
安全：使用虛構資料，不提交API key、學生個資、成績或機密。

## 🤝 同儕覆核
交換input與測試表，確認每個角色的內容來源、裁切後仍為完整配對、失敗不留下半套state。
找一項未驗證或失敗條件，修改一個主要變因後重測並記錄原因。

## 📚 官方文件與版本使用
- [Conversation state](https://developers.openai.com/api/docs/guides/conversation-state)
- [Streaming responses](https://developers.openai.com/api/docs/guides/streaming-responses)
- [Responses API](https://developers.openai.com/api/reference/resources/responses/methods/create)
模型沿用OPENAI_MODEL／gpt-5.4-mini；本次沒有用教學帳號驗證可用性，課前須試跑。
本修訂版與原教材並存，課堂選定一組使用，不要求學生交錯執行兩版。